# 🥉 Bronze Layer: Ingestión de Datos - Single Ticker

Este notebook ingesta datos históricos de **un solo ticker** desde DataBursátil API v2.

## 📋 Flujo del Proceso

1. **Setup**: Importar librerías
2. **Configuración**: Definir ticker y parámetros
3. **API Call**: Llamar a `/v2/historicos`
4. **Guardar**: Almacenar JSON raw en `bronze.api_responses`
5. **Verificar**: Confirmar que se guardó correctamente

## 🎯 Para Usar

1. Cambia `TICKER_TO_PROCESS` en la celda de configuración
2. Ajusta `LOOKBACK_DAYS` si necesitas más/menos histórico
3. Run All (Ctrl+Shift+Enter)

## ⚠️ Requisitos Previos

- Ejecutar `00_create_schemas_and_tables.ipynb` primero
- Secret configurado: `databursatil/api_token`
- Cluster Databricks con Delta Lake habilitado

## 📦 Setup: Librerías y Spark

In [ ]:
import requests
import json
import uuid
from datetime import datetime, timezone, timedelta
#changes

# Para Databricks
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

print("✓ Librerías importadas")

## ⚙️ Configuración del Ticker

**AQUÍ defines qué ticker procesar**

In [ ]:
# ============================================
# PASO 1: Verificar formato del ticker en catálogo
# ============================================

print("🔍 Buscando WALMEX en el catálogo...\n")

# Buscar en el catálogo cómo viene WALMEX
catalog_check = spark.sql("""
    SELECT 
        emisora,
        serie, 
        emisora_serie,
        isin,
        is_active 
    FROM silver.tickers 
    WHERE UPPER(emisora) LIKE '%WALMEX%' 
       OR UPPER(emisora_serie) LIKE '%WALMEX%'
    LIMIT 10
""")

print("📋 Resultados encontrados:")
display(catalog_check)

print("\n💡 IMPORTANTE: Copia el valor de la columna 'emisora_serie' y úsalo en la siguiente celda")
print("   Ejemplo: Si aparece 'WALMEX*', ese es el valor correcto a usar")

## 🔧 Configuración del Ticker (después de verificar)

## 🧪 Prueba rápida de la API (opcional)

Puedes ejecutar esto para probar si el ticker funciona antes de hacer la ingestión completa:

## 🌐 Llamada a la API

In [ ]:
# Calcular fechas
end_date = datetime.now(timezone.utc).date()
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

print(f"📅 Rango: {start_date} → {end_date}\n")

# Construir URL de la API
base_url = "https://api.databursatil.com/v2/historicos"
params = {
    "token": TOKEN,
    "emisora_serie": TICKER_TO_PROCESS,
    "inicio": start_date.isoformat(),
    "final": end_date.isoformat()
}

# Hacer request
print(f"🌐 Llamando API para {TICKER_TO_PROCESS}...")
print(f"   URL: {base_url}")
print(f"   Params: emisora_serie={TICKER_TO_PROCESS}, inicio={start_date}, final={end_date}\n")

try:
    response = requests.get(base_url, params=params, timeout=30)
    
    if response.status_code == 200:
        data = response.json()
        print(f"✅ Éxito! Recibidos {len(data)} registros")
        print(f"   Tamaño respuesta: {len(response.text)} bytes\n")
        
        # Mostrar primeros 3 registros
        print("📊 Primeros registros:")
        for i, (fecha, valores) in enumerate(list(data.items())[:3]):
            print(f"   {fecha}: precio={valores.get('precio')}, importe={valores.get('importe')}")
        
        api_response_json = response.text
        api_success = True
        api_error = None
        
    else:
        print(f"❌ Error HTTP {response.status_code}")
        print(f"   Respuesta: {response.text[:200]}")
        api_response_json = response.text
        api_success = False
        api_error = f"HTTP {response.status_code}"
        
except Exception as e:
    print(f"❌ Excepción: {str(e)}")
    api_response_json = None
    api_success = False
    api_error = str(e)

## 💾 Guardar en Bronze Layer

In [ ]:
# Crear registro para bronze.api_responses
record = {
    "response_id": str(uuid.uuid4()),
    "endpoint": "/v2/historicos",
    "request_params": {
        "emisora_serie": TICKER_TO_PROCESS,
        "inicio": start_date.isoformat(),
        "final": end_date.isoformat()
    },
    "response_body": api_response_json,
    "response_status": response.status_code if 'response' in locals() else None,
    "response_headers": dict(response.headers) if 'response' in locals() else {},
    "ingestion_timestamp": datetime.now(timezone.utc),
    "data_date": end_date,
    "ticker": TICKER_TO_PROCESS,
    "is_successful": api_success,
    "error_message": api_error
}

print(f"\n📝 Registro creado:")
print(f"   response_id: {record['response_id']}")
print(f"   ticker: {record['ticker']}")
print(f"   is_successful: {record['is_successful']}")
print(f"   data_date: {record['data_date']}")

# EN DATABRICKS: Guardar en Delta Lake
df = spark.createDataFrame([record])
df.write.format("delta").mode("append").saveAsTable("bronze.api_responses")
print("✅ Guardado en bronze.api_responses")

# LOCAL: Guardar como JSON (comentado para Databricks)
"""
import os
os.makedirs("output", exist_ok=True)
with open(f"output/{TICKER_TO_PROCESS.replace('*', '')}_bronze.json", "w") as f:
    json.dump(record, f, indent=2, default=str)
print(f"✅ Guardado en output/{TICKER_TO_PROCESS.replace('*', '')}_bronze.json")
"""

## ✅ Verificación

In [ ]:
# Verificar que se guardó
verification_query = f"""
    SELECT 
        response_id,
        ticker,
        endpoint,
        is_successful,
        data_date,
        ingestion_timestamp
    FROM bronze.api_responses
    WHERE ticker = '{TICKER_TO_PROCESS}'
    ORDER BY ingestion_timestamp DESC
    LIMIT 5
"""

display(spark.sql(verification_query))

In [ ]:
# ============================================
# PASO 2: Configurar con el ticker EXACTO del catálogo
# ============================================

# 🎯 PEGA AQUÍ el valor exacto de 'emisora_serie' que viste arriba
TICKER_TO_PROCESS = "WALMEX*"  # ← Cambia esto según lo que aparezca en el catálogo

LOOKBACK_DAYS = 365  # 1 año de histórico

# En Databricks, usar secrets:
TOKEN = dbutils.secrets.get(scope="databursatil", key="api_token")

# Para testing local:
# TOKEN = "763d9cdce4895c32fcb2f63358c6de"

print(f"✅ Configuración:")
print(f"   🎯 Ticker: {TICKER_TO_PROCESS}")
print(f"   📅 Días históricos: {LOOKBACK_DAYS}")
print(f"   🔑 Token: {TOKEN[:10]}...")

# Verificar que el ticker existe en el catálogo
ticker_exists = spark.sql(f"""
    SELECT COUNT(*) as count 
    FROM silver.tickers 
    WHERE emisora_serie = '{TICKER_TO_PROCESS}' AND is_active = true
""").collect()[0]['count']

if ticker_exists > 0:
    print(f"\n✅ Ticker '{TICKER_TO_PROCESS}' encontrado en catálogo y está activo")
else:
    print(f"\n⚠️ ADVERTENCIA: Ticker '{TICKER_TO_PROCESS}' NO encontrado en catálogo")
    print("   Revisa la celda anterior y copia el valor exacto de 'emisora_serie'")

In [ ]:
# Prueba rápida: obtener solo los últimos 7 días
test_end = datetime.now(timezone.utc).date()
test_start = test_end - timedelta(days=7)

test_url = f"https://api.databursatil.com/v2/historicos?token={TOKEN}&emisora_serie={TICKER_TO_PROCESS}&inicio={test_start.isoformat()}&final={test_end.isoformat()}"

print(f"🧪 Probando API con {TICKER_TO_PROCESS} (últimos 7 días)...")
print(f"   URL: https://api.databursatil.com/v2/historicos")
print(f"   Params: emisora_serie={TICKER_TO_PROCESS}, inicio={test_start}, final={test_end}\n")

test_response = requests.get(test_url, timeout=30)

if test_response.status_code == 200:
    test_data = test_response.json()
    print(f"✅ ¡Éxito! La API acepta el ticker '{TICKER_TO_PROCESS}'")
    print(f"   Recibidos {len(test_data)} días de datos")
    print(f"\n   Primeros registros:")
    for fecha, valores in list(test_data.items())[:3]:
        print(f"      {fecha}: precio={valores.get('precio')}, importe={valores.get('importe')}")
    print("\n✅ Puedes continuar con la siguiente celda para ingestar todo el histórico")
else:
    print(f"❌ Error {test_response.status_code}: {test_response.text[:300]}")
    print("\n💡 Si dice 'Emisora desconocida':")
    print("   1. Regresa a la celda anterior")
    print("   2. Verifica el valor exacto de 'emisora_serie' en el catálogo")
    print("   3. Cópialo exactamente como aparece (con o sin asterisco)")